# Exercise 2: Filtering

Exercise: How can we use intensity based filtering to remove background?

1. Execute the following code to load the example image `image_nuclei_noisy` and display it.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import skimage
import skimage.morphology
import skimage.filters
import stackview

image_nuclei = skimage.io.imread('https://cildata.crbs.ucsd.edu/media/images/13585/13585.tif')[0:500,0:500,2]
image_nuclei_noisy = image_nuclei // 2
image_nuclei_noisy = image_nuclei_noisy + 50 * np.linspace(0, 1, image_nuclei_noisy.shape[1]) * np.ones(image_nuclei_noisy.shape)
image_nuclei_noisy = (image_nuclei_noisy - image_nuclei_noisy.min()) / (image_nuclei_noisy.max() - image_nuclei_noisy.min())
image_nuclei_noisy = skimage.util.random_noise(image_nuclei_noisy, mode='s&p')
image_nuclei_noisy = (image_nuclei_noisy * 255).astype(np.uint8)

plt.imshow(image_nuclei_noisy, cmap='gray')
plt.colorbar()
plt.axis('off')
plt.show()

URLError: <urlopen error [Errno 11001] getaddrinfo failed>

2. Try segmenting the image `image_nuclear_noisy` using a threshold. What do you observe?

In [ ]:
threshold_otsu = skimage.filters.threshold_otsu(image_nuclei_noisy)

stackview.switch([image_nuclei_noisy, image_nuclei_noisy > threshold_otsu])

3. Find a way to perform local background subtraction on the image using filtering.
   * Option 1: Find a filter to estimate the image background and subtract it from the original image
   * Option 2: Test one of the options suggested in [this article](https://biapol.github.io/blog/ryan_savill/03_background_subtraction/readme.html)

In [ ]:
# option 1
background = skimage.filters.rank.percentile(image_nuclei_noisy, p0=0.1, footprint=skimage.morphology.disk(100))
background = skimage.filters.gaussian(background, sigma=50, preserve_range=True)

image_bs = image_nuclei_noisy - background
image_bs = np.abs(image_bs) # remove negative values

# visualise side by side
fig, axs = plt.subplots(1, 3, figsize=(10,6))

axs[0].imshow(image_nuclei_noisy, cmap='gray')
axs[0].set_title('Raw image')
axs[0].axis('off')

axs[1].imshow(background, cmap='gray')
axs[1].set_title('Estimated background')
axs[1].axis('off')

axs[2].imshow(image_bs, cmap='gray')
axs[2].set_title('Raw image - background')
axs[2].axis('off')

plt.show()

In [ ]:
# option 2
image_bs1 = skimage.filters.difference_of_gaussians(image_nuclei_noisy, 1, 50)

# visualise side by side
fig, axs = plt.subplots(1, 2, figsize=(10,6))

axs[0].imshow(image_nuclei_noisy, cmap='gray')
axs[0].set_title('Raw image')
axs[0].axis('off')

axs[1].imshow(image_bs1, cmap='gray')
axs[1].set_title('Difference of Gaussians')
axs[1].axis('off')

plt.show()

4. Try segmenting the background subtract image using a threshold.

In [ ]:
stackview.switch([image_bs, image_bs > skimage.filters.threshold_otsu(image_bs)])

5. Denoise the background subtracted image using image filtering.

In [ ]:
image_denoised = skimage.filters.median(image_bs, footprint=skimage.morphology.disk(3))

# visualise side by side
fig, axs = plt.subplots(1, 3, figsize=(12,8))

axs[0].imshow(image_nuclei_noisy, cmap='gray')
axs[0].set_title('Raw image')
axs[0].axis('off')

axs[1].imshow(image_bs, cmap='gray')
axs[1].set_title('Background subtracted')
axs[1].axis('off')

axs[2].imshow(image_denoised, cmap='gray')
axs[2].set_title('Background subtracted and denoised')
axs[2].axis('off')

plt.show()

6. Try segmenting the denoised and background subtract image using a threshold.

In [ ]:
image_denoised_th = image_denoised > skimage.filters.threshold_otsu(image_denoised)

stackview.switch([image_denoised, image_denoised_th])

7. Visualize the steps of the exercise including
    - the original noisy image
    - the background subtracted image
    - the denoised image
    - the thresholded denoised image

In [ ]:
stackview.switch([image_nuclei_noisy, image_bs, image_denoised, image_denoised_th])

8. Instead of the background subtracted and denoised image, threshold the image for which only background has been subtracted. Try to eliminate small objects using morphological operations. 

In [ ]:
binary = image_bs > skimage.filters.threshold_otsu(image_bs)
binary_open = skimage.morphology.isotropic_opening(binary, radius=1)
binary_close = skimage.morphology.isotropic_closing(binary_open, radius=1)

# visualise side by side
fig, axs = plt.subplots(1, 3, figsize=(12,8))

axs[0].imshow(binary, cmap='gray')
axs[0].set_title('Binary mask')
axs[0].axis('off')

axs[1].imshow(binary_open, cmap='gray')
axs[1].set_title('After morphological opening')
axs[1].axis('off')

axs[2].imshow(binary_close, cmap='gray')
axs[2].set_title('After morphological opening + closing')
axs[2].axis('off')

plt.show()